# Example 6.4.1 / 1: Test a Policy Constraint, Then Switch Objectives

Companion notebook for **6.4.1 Dropping and Restoring Constraints** in the AMPL lab Documentation, section "Test a Policy Constraint, Then Switch Objectives". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Test a Policy Constraint, Then Switch Objectives

Suppose a production model contains a minimum-service rule and two objectives, cost and CO2
emissions. With several objectives declared, AMPL minimizes the first one, `Cost`, until told
otherwise. The drop of one member is sent with `ampl.eval()`; see the warning below for why.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                                          # products
param demand {PROD} >= 0;                          # units demanded
param cost {PROD} >= 0;                                # cost per unit made
param co2 {PROD} >= 0;                                 # CO2 emitted per unit made
var Make {p in PROD} >= 0;                             # units made
minimize Cost: sum {p in PROD} cost[p] * Make[p];      # total production cost
minimize Emissions: sum {p in PROD} co2[p] * Make[p];  # total CO2 emitted
subject to Service {p in PROD}: Make[p] >= 0.8 * demand[p];  # policy: serve at least 80% of demand

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["demand"] = {"bands": 10, "coils": 20, "plate": 5}
ampl.param["cost"] = {"bands": 2, "coils": 3, "plate": 4}
ampl.param["co2"] = {"bands": 3, "coils": 1, "plate": 2}

In [5]:
# run.py
ampl.option["solver_msg"] = 0
ampl.solve(solver="highs")
print("all rules:", ampl.obj["Cost"].value())

ampl.eval('drop Service["plate"];')
ampl.solve(solver="highs")
print("without plate:", ampl.obj["Cost"].value(), ampl.var["Make"].to_dict())

ampl.eval('restore Service["plate"];')
ampl.solve(solver="highs")
print("restored:", ampl.obj["Cost"].value())

all rules: 80.0
without plate: 64.0 {'bands': 8, 'coils': 16, 'plate': 0}
restored: 80.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
param: demand cost co2 :=
  bands 10 2 3
  coils 20 3 1
  plate  5 4 2;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
solve;
display Cost;
drop Service["plate"];
solve;
display Cost, Make;
restore Service["plate"];
solve;
display Cost;

Cost = 80

Cost = 64

Make [*] :=
bands   8
coils  16
plate   0
;

Cost = 80



### Another way in amplpy

Without the `plate` rule the cost falls from 80 to 64, the 16 that producing 4 units of `plate`
cost. That is a controlled counterfactual, not proof that the rule is unnecessary.

> **Pitfall: `drop()` on one member drops them all**
> In amplpy 0.17, `ampl.con["Service"]["plate"].drop()` drops every member of `Service` instead
> of just `plate`, and `restore()` likewise acts on all of them. Send a subscripted `drop` or `restore`
> with `ampl.eval()`, and check `Service.astatus` when in doubt: a dropped member shows `drop`.

With several declared objectives, `objective Emissions;` is equivalent to dropping all
objectives and then restoring `Emissions`. Selecting the objective explicitly is safer than
relying on whichever one happens to be active. amplpy has no method for the `objective` command,
so the Python tab sends it with `ampl.eval()`. With the same `model.mod` and data:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["demand"] = {"bands": 10, "coils": 20, "plate": 5}
ampl.param["cost"] = {"bands": 2, "coils": 3, "plate": 4}
ampl.param["co2"] = {"bands": 3, "coils": 1, "plate": 2}

In [11]:
# run.py
ampl.option["solver_msg"] = 0
for name in ["Cost", "Emissions"]:
    ampl.eval(f"objective {name};")
    ampl.solve(solver="highs")
    print(name, ampl.obj[name].value())

Cost 80.0
Emissions 48.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod` and `data.dat`:

In [12]:
%%writefile data.dat
set PROD := bands coils plate;
param: demand cost co2 :=
  bands 10 2 3
  coils 20 3 1
  plate  5 4 2;

Overwriting data.dat


In [13]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [14]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
objective Cost;
solve;
display Cost;
objective Emissions;
solve;
display Emissions;

Cost = 80

Emissions = 48

